# Buổi 1 — Layout & Text Detection

**Goal:** benchmark DB-Net (pretrained) vs MixNet (placeholder for fine-tuned) on SROIE test images.

**Output:** `reports/benchmarks/detection_results.json` + `reports/error_analysis/detection.md` + 10 overlay PNGs.

## Setup


In [ ]:
# Make the package importable when the notebook is launched from notebooks/
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())


In [ ]:
from src.data.sroie_loader import list_ids, iter_split
from src.detection import DBNetDetector, MixNetDetector, evaluate_image, aggregate, draw_overlay
print('SROIE test images available:', len(list_ids('test')))


## Theory recap

- **DocLayout-YOLO** — YOLO-based page region segmentation (text / table / figure / title). Not benchmarked here; covered as theory in S1.
- **DB-Net** — differentiable binarization, predicts a probability map + threshold map; fast and the standard baseline.
- **MixNet** — mixed depth-wise convolutions + multi-scale feature fusion; designed for challenging scene text (curved, blurred). The lab fine-tunes it on SROIE; until that checkpoint exists, `MixNetDetector` falls back to DocTR's FAST so the benchmark scaffold is testable.

## Run the benchmark

Curriculum says **50 SROIE test images**. The driver below mirrors `python -m src.detection.run_benchmark --limit 50`.


In [ ]:
LIMIT = 50

dbnet  = DBNetDetector()
mixnet = MixNetDetector()  # falls back to FAST until mixnet_sroie_finetuned.pth exists
detectors = [dbnet, mixnet]


In [ ]:
import time
from src.detection import AggregateStats

results = {}
samples = {}  # keep 10 (image, preds, gt) tuples per detector for visualization

for det in detectors:
    per05, per03, sample_buf = [], [], []
    t0 = time.time()
    for rid, img, gt in iter_split('test', limit=LIMIT):
        preds = det.detect(img)
        per05.append(evaluate_image(preds, gt, iou_thresh=0.5))
        per03.append(evaluate_image(preds, gt, iou_thresh=0.3))
        if len(sample_buf) < 10:
            sample_buf.append((rid, img, preds, gt))
    agg05 = aggregate(per05)
    agg03 = aggregate(per03)
    elapsed = time.time() - t0
    results[det.name] = {
        'n_images': agg05.n_images,
        'elapsed_s': round(elapsed, 1),
        'fps': round(agg05.n_images / elapsed, 2) if elapsed else 0.0,
        'iou_0.5': {'P': agg05.precision, 'R': agg05.recall, 'F1': agg05.f1,
                    'tp': agg05.tp, 'fp': agg05.fp, 'fn': agg05.fn},
        'iou_0.3': {'P': agg03.precision, 'R': agg03.recall, 'F1': agg03.f1,
                    'tp': agg03.tp, 'fp': agg03.fp, 'fn': agg03.fn},
    }
    samples[det.name] = sample_buf
    print(f'{det.name:20s}  IoU=.5 F1={agg05.f1:.3f}  IoU=.3 F1={agg03.f1:.3f}  {results[det.name]["fps"]} fps')


## Comparison table


In [ ]:
import pandas as pd
rows = []
for name, r in results.items():
    for iou in ('0.5', '0.3'):
        m = r[f'iou_{iou}']
        rows.append({'detector': name, 'IoU': iou,
                     'P': round(m['P'], 3), 'R': round(m['R'], 3), 'F1': round(m['F1'], 3),
                     'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'],
                     'FPS': r['fps']})
pd.DataFrame(rows)


## Visualize 10 sample overlays

Green = ground truth (SROIE line-level). Orange = detector prediction (word-level for DB-Net / FAST).


In [ ]:
import matplotlib.pyplot as plt

for name, buf in samples.items():
    fig, axes = plt.subplots(2, 5, figsize=(18, 10))
    fig.suptitle(name, fontsize=14)
    for ax, (rid, img, preds, gt) in zip(axes.ravel(), buf):
        ax.imshow(draw_overlay(img, preds, gt))
        ax.set_title(f'{rid}  gt={len(gt)} pred={len(preds)}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## Granularity caveat

SROIE GT is **line-level**, DocTR predicts **word-level** boxes → per-pair IoU is structurally low (one GT line contains many word predictions). That depresses both P (many FPs per GT line) and F1@0.5. The IoU=0.3 row is the more honest *relative* metric until either:

1. predicted words are clustered into lines, or
2. MixNet is fine-tuned on SROIE's line-level boxes (the actual Buổi 1 lab).

## Save artifacts

Writes the standalone outputs that the curriculum expects (JSON results + markdown report).


In [ ]:
import json
from pathlib import Path
out = Path('reports/benchmarks/detection_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    json.dump({'limit': LIMIT, 'detectors': [{**{'detector': k}, **v} for k, v in results.items()]}, f, indent=2, default=float)
print('wrote', out)


## Next steps (Buổi 1 lab)

- Vendor the official MixNet architecture under `src/detection/mixnet_arch.py`.
- Build the polygon-bbox training set from `data/sroie/annotations/train/`.
- Train with lr warmup + augmentation; save to `checkpoints/detection/mixnet_sroie_finetuned.pth`.
- Re-run this notebook; `MixNetDetector(checkpoint=…)` will load the real weights and the comparison becomes meaningful (line-level vs line-level).
